# Predição de curvas de pressão de poço — versão final com 5 repetições

Esta versão preserva a estrutura do experimento anterior e altera **somente a avaliação final de 10 para 5 repetições**, usando as mesmas sementes para todos os modelos: **42, 43, 44, 45 e 46**.

- **Random Forest, XGBoost e MLP:** otimização bayesiana com Optuna/TPE, total de 50 trials por modelo.
- **TabPFN-3:** mantém o procedimento anterior, **sem Optuna**, e seus resultados reais são importados de CSV.
- Validação interna temporal: últimos 20% de cada curva.
- Amostragem da otimização: até 70.000 linhas de treino e 20.000 de validação.
- Métricas finais: RMSE, MAE, R², RMSE médio e mediano por curva.

> **Importante:** se o seu orientador exigir Optuna também no TabPFN, este notebook não representa esse novo experimento. Nesse caso o TabPFN precisaria ser otimizado e executado novamente.

## 1. Dependências

In [ ]:
# Execute esta célula apenas se as bibliotecas ainda não estiverem instaladas no ambiente .venv.
import sys
print("Python em uso:", sys.version)

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "Use o ambiente Python 3.11 (.venv) preparado para este projeto."
    )

# Se o seu ambiente já foi configurado anteriormente, pode comentar as linhas %pip.
%pip -q install -U numpy pandas matplotlib scikit-learn optuna xgboost openpyxl tensorflow
print("Dependências verificadas/instaladas.")

## 2. Imports e configurações

In [ ]:
import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.trial import TrialState

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================================================
# CONFIGURAÇÕES DO EXPERIMENTO
# ============================================================
SEED_BASE = 42
N_REPETITIONS = 5
SEEDS_RUNS = list(range(SEED_BASE, SEED_BASE + N_REPETITIONS))

TARGET = "R_pressure"
GROUP_COL = "curve_id"
ORDER_COL = "t_index"
TIME_COL = "Q_time"

VAL_FRACTION = 0.20

# Optuna: total desejado de trials por estudo.
N_TRIALS_RF = 50
N_TRIALS_XGB = 50
N_TRIALS_MLP = 50

# Amostragem usada apenas durante a otimização.
TUNE_MAX_TRAIN_ROWS = 70000
TUNE_MAX_VAL_ROWS = 20000

# Detecta GPU para XGBoost. A MLP usa o TensorFlow disponível no sistema.
TF_HAS_GPU = len(tf.config.list_physical_devices("GPU")) > 0
try:
    import torch
    TORCH_HAS_CUDA = bool(torch.cuda.is_available())
    GPU_NAME = torch.cuda.get_device_name(0) if TORCH_HAS_CUDA else None
except Exception:
    TORCH_HAS_CUDA = False
    GPU_NAME = None

XGB_DEVICE = "cuda" if TORCH_HAS_CUDA else "cpu"

print("TensorFlow - GPU detectada:", TF_HAS_GPU)
print("PyTorch/CUDA - GPU detectada:", TORCH_HAS_CUDA)
if GPU_NAME:
    print("GPU:", GPU_NAME)
print("XGBoost device:", XGB_DEVICE)
print("Número de repetições finais:", N_REPETITIONS)
print("Seeds finais:", SEEDS_RUNS)

## 3. Localização e carregamento dos dados

In [ ]:
# Localiza a pasta dos dados sem vasculhar a .venv.
BASE_DIR = Path.cwd()

candidate_dirs = [
    BASE_DIR,
    BASE_DIR / "Projeto_Machine_Learning_VSCode",
    BASE_DIR.parent,
]

PROJECT_DIR = None
for folder in candidate_dirs:
    if (folder / "train_timeseries.xlsx").exists() and (folder / "test_timeseries.xlsx").exists():
        PROJECT_DIR = folder.resolve()
        break

# Também aceita nomes como train_timeseries(1).xlsx.
if PROJECT_DIR is None:
    for folder in candidate_dirs:
        train_candidates = list(folder.glob("*train*timeseries*.xlsx")) if folder.exists() else []
        test_candidates = list(folder.glob("*test*timeseries*.xlsx")) if folder.exists() else []
        if train_candidates and test_candidates:
            PROJECT_DIR = folder.resolve()
            break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Não encontrei train_timeseries.xlsx e test_timeseries.xlsx. "
        "Coloque-os na pasta do projeto ou na pasta imediatamente acima."
    )

train_candidates = sorted(PROJECT_DIR.glob("*train*timeseries*.xlsx"))
test_candidates = sorted(PROJECT_DIR.glob("*test*timeseries*.xlsx"))
TRAIN_PATH = train_candidates[0]
TEST_PATH = test_candidates[0]

print("Pasta dos dados:", PROJECT_DIR)
print("Treino:", TRAIN_PATH.name)
print("Teste :", TEST_PATH.name)

train_raw = pd.read_excel(TRAIN_PATH)
test_raw = pd.read_excel(TEST_PATH)

for df in (train_raw, test_raw):
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

required = {TARGET, GROUP_COL, ORDER_COL, TIME_COL}
missing_train = required - set(train_raw.columns)
missing_test = required - set(test_raw.columns)
if missing_train:
    raise ValueError(f"Colunas obrigatórias ausentes no treino: {missing_train}")
if missing_test:
    raise ValueError(f"Colunas obrigatórias ausentes no teste: {missing_test}")
if list(train_raw.columns) != list(test_raw.columns):
    raise ValueError("Treino e teste possuem colunas diferentes ou em ordem diferente.")

print("Treino:", train_raw.shape)
print("Teste :", test_raw.shape)
print("Curvas no treino:", train_raw[GROUP_COL].nunique())
print("Curvas no teste :", test_raw[GROUP_COL].nunique())

display(pd.DataFrame({
    "ausentes_treino": train_raw.isna().sum(),
    "percentual_treino": 100 * train_raw.isna().mean(),
    "ausentes_teste": test_raw.isna().sum(),
    "percentual_teste": 100 * test_raw.isna().mean(),
}).sort_values("ausentes_treino", ascending=False))

## 4. Verificação temporal e engenharia de atributos

In [ ]:
# O conjunto de teste deve ser temporalmente posterior ao conjunto de treino em cada curva.
train_max_t = train_raw.groupby(GROUP_COL)[ORDER_COL].max()
test_min_t = test_raw.groupby(GROUP_COL)[ORDER_COL].min()
common_curves = train_max_t.index.intersection(test_min_t.index)
violations = (test_min_t.loc[common_curves] <= train_max_t.loc[common_curves]).sum()

print("Curvas presentes nos dois arquivos:", len(common_curves))
print("Sobreposições temporais encontradas:", int(violations))
if violations:
    raise ValueError("Há pontos do teste que não são posteriores aos pontos de treino.")


def engineer_known_features(train_df, test_df):
    """Cria apenas atributos conhecidos no instante da previsão; nunca usa R_pressure como preditor."""
    a = train_df.copy()
    b = test_df.copy()
    a["__split"] = "train"
    b["__split"] = "test"
    a["__row_order"] = np.arange(len(a))
    b["__row_order"] = np.arange(len(b))

    full = pd.concat([a, b], ignore_index=True)
    full = full.sort_values([GROUP_COL, ORDER_COL], kind="mergesort")

    q = full[TIME_COL].astype(float)
    full["Q_time_signed_log"] = np.sign(q) * np.log1p(np.abs(q))
    full["t_index_log"] = np.log1p(full[ORDER_COL].clip(lower=0))

    full["delta_Q_time"] = full.groupby(GROUP_COL, sort=False)[TIME_COL].diff()
    dq = full["delta_Q_time"].astype(float)
    full["delta_Q_time_signed_log"] = np.sign(dq) * np.log1p(np.abs(dq))

    full["Q_time_from_start"] = (
        full[TIME_COL]
        - full.groupby(GROUP_COL, sort=False)[TIME_COL].transform("first")
    )

    train_out = (
        full[full["__split"] == "train"]
        .sort_values("__row_order")
        .drop(columns=["__split", "__row_order"])
        .reset_index(drop=True)
    )
    test_out = (
        full[full["__split"] == "test"]
        .sort_values("__row_order")
        .drop(columns=["__split", "__row_order"])
        .reset_index(drop=True)
    )
    return train_out, test_out


train_df, test_df = engineer_known_features(train_raw, test_raw)
FEATURES = [c for c in train_df.columns if c not in {TARGET, GROUP_COL}]

print("Quantidade de variáveis preditoras:", len(FEATURES))
print(FEATURES)

## 5. Validação temporal interna e amostragem balanceada

In [ ]:
def temporal_tail_split(df, val_fraction=0.20):
    """Reserva os últimos 20% de cada curva para validação interna."""
    ordered = df.sort_values([GROUP_COL, ORDER_COL], kind="mergesort")
    position = ordered.groupby(GROUP_COL, sort=False).cumcount()
    size = ordered.groupby(GROUP_COL, sort=False)[GROUP_COL].transform("size")

    n_val = np.ceil(size * val_fraction).astype(int).clip(lower=1)
    is_val = position >= (size - n_val)

    train_idx = ordered.index[~is_val].to_numpy()
    val_idx = ordered.index[is_val].to_numpy()
    return train_idx, val_idx


def balanced_group_sample(index_array, groups, max_rows, seed):
    """Amostragem aproximadamente balanceada entre curvas."""
    index_array = np.asarray(index_array)
    if max_rows is None or len(index_array) <= max_rows:
        return index_array

    rng = np.random.default_rng(seed)
    temp = pd.DataFrame({
        "idx": index_array,
        "group": groups.loc[index_array].to_numpy(),
    })
    unique_groups = temp["group"].unique()
    per_group = max(1, max_rows // len(unique_groups))

    selected = []
    for _, part in temp.groupby("group", sort=False):
        n = min(len(part), per_group)
        selected.extend(
            rng.choice(part["idx"].to_numpy(), size=n, replace=False).tolist()
        )

    selected = np.array(selected, dtype=int)
    if len(selected) < max_rows:
        remaining = np.setdiff1d(index_array, selected, assume_unique=False)
        extra_n = min(max_rows - len(selected), len(remaining))
        if extra_n > 0:
            selected = np.concatenate([
                selected,
                rng.choice(remaining, size=extra_n, replace=False),
            ])
    elif len(selected) > max_rows:
        selected = rng.choice(selected, size=max_rows, replace=False)

    return np.sort(selected)


inner_train_idx, inner_val_idx = temporal_tail_split(train_df, VAL_FRACTION)

X_all = train_df[FEATURES]
y_all = train_df[TARGET].astype(float)
groups_all = train_df[GROUP_COL].astype(int)

X_test = test_df[FEATURES]
y_test = test_df[TARGET].astype(float)
groups_test = test_df[GROUP_COL].astype(int)

opt_train_idx = balanced_group_sample(
    inner_train_idx, groups_all, TUNE_MAX_TRAIN_ROWS, SEED_BASE
)
opt_val_idx = balanced_group_sample(
    inner_val_idx, groups_all, TUNE_MAX_VAL_ROWS, SEED_BASE
)

X_opt_train = X_all.loc[opt_train_idx]
y_opt_train = y_all.loc[opt_train_idx]
X_opt_val = X_all.loc[opt_val_idx]
y_opt_val = y_all.loc[opt_val_idx]
g_opt_val = groups_all.loc[opt_val_idx]

print("Treino interno completo:", len(inner_train_idx))
print("Validação interna completa:", len(inner_val_idx))
print("Treino usado no Optuna:", X_opt_train.shape)
print("Validação usada no Optuna:", X_opt_val.shape)

## 6. Métricas, seeds e armazenamento do Optuna

In [ ]:
def regression_metrics(y_true, y_pred, groups=None):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    metrics = {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }

    if groups is not None:
        temp = pd.DataFrame({
            "erro_quadrado": (y_true - y_pred) ** 2,
            "group": np.asarray(groups),
        })
        curve_rmse = np.sqrt(temp.groupby("group")["erro_quadrado"].mean())
        metrics["RMSE_medio_por_curva"] = float(curve_rmse.mean())
        metrics["RMSE_mediano_por_curva"] = float(curve_rmse.median())
    return metrics


def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


# Usa o mesmo banco do experimento anterior. O helper abaixo evita adicionar
# mais 50 trials se o estudo já tiver 50 trials completos.
STORAGE_PATH = PROJECT_DIR / "optuna_pressao_poco.db"
STORAGE = f"sqlite:///{STORAGE_PATH.as_posix()}"


def optimize_until_total(study, objective, total_trials):
    completed = sum(t.state == TrialState.COMPLETE for t in study.trials)
    remaining = max(0, total_trials - completed)
    print(f"Estudo {study.study_name}: {completed} trials completos; faltam {remaining} para totalizar {total_trials}.")
    if remaining > 0:
        study.optimize(
            objective,
            n_trials=remaining,
            gc_after_trial=True,
            show_progress_bar=True,
        )
    return study

print("Banco Optuna:", STORAGE_PATH)

## 7. Random Forest + Optuna

In [ ]:
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 900, step=100),
        "max_depth": trial.suggest_int("max_depth", 8, 35),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_float("max_features", 0.35, 1.0),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "random_state": SEED_BASE,
        "n_jobs": -1,
    }
    if params["bootstrap"]:
        params["max_samples"] = trial.suggest_float("max_samples", 0.65, 1.0)

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", RandomForestRegressor(**params)),
    ])
    model.fit(X_opt_train, y_opt_train)
    pred = model.predict(X_opt_val)
    return float(np.sqrt(mean_squared_error(y_opt_val, pred)))


study_rf = optuna.create_study(
    study_name="rf_pressao_poco",
    direction="minimize",
    sampler=TPESampler(seed=SEED_BASE),
    storage=STORAGE,
    load_if_exists=True,
)
study_rf = optimize_until_total(study_rf, objective_rf, N_TRIALS_RF)

print("Melhor RMSE RF:", study_rf.best_value)
print("Melhores parâmetros RF:")
print(study_rf.best_params)

## 8. XGBoost + Optuna

In [ ]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 2200, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 25.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.60, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
        "max_bin": trial.suggest_categorical("max_bin", [256, 512]),
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": XGB_DEVICE,
        "random_state": SEED_BASE,
        "n_jobs": -1,
        "early_stopping_rounds": 60,
    }

    model = XGBRegressor(**params)
    model.fit(
        X_opt_train,
        y_opt_train,
        eval_set=[(X_opt_val, y_opt_val)],
        verbose=False,
    )
    pred = model.predict(X_opt_val)
    if getattr(model, "best_iteration", None) is not None:
        trial.set_user_attr("best_iteration", int(model.best_iteration))
    return float(np.sqrt(mean_squared_error(y_opt_val, pred)))


study_xgb = optuna.create_study(
    study_name="xgb_pressao_poco",
    direction="minimize",
    sampler=TPESampler(seed=SEED_BASE),
    storage=STORAGE,
    load_if_exists=True,
)
study_xgb = optimize_until_total(study_xgb, objective_xgb, N_TRIALS_XGB)

print("Melhor RMSE XGBoost:", study_xgb.best_value)
print("Melhores parâmetros XGBoost:")
print(study_xgb.best_params)
print("Melhor iteração:", study_xgb.best_trial.user_attrs.get("best_iteration"))

## 9. MLP + Optuna

In [ ]:
mlp_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
X_mlp_train = mlp_preprocessor.fit_transform(X_opt_train)
X_mlp_val = mlp_preprocessor.transform(X_opt_val)


def build_mlp(params, input_dim, seed):
    set_all_seeds(seed)
    model = keras.Sequential([layers.Input(shape=(input_dim,))])
    for i in range(params["n_layers"]):
        units = params[f"units_{i}"]
        model.add(layers.Dense(
            units,
            activation=params["activation"],
            kernel_regularizer=regularizers.l2(params["l2"]),
        ))
        if params["batch_norm"]:
            model.add(layers.BatchNormalization())
        if params["dropout"] > 0:
            model.add(layers.Dropout(params["dropout"]))
    model.add(layers.Dense(1))

    optimizer = keras.optimizers.Adam(learning_rate=params["learning_rate"])
    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=[keras.metrics.RootMeanSquaredError()],
    )
    return model


def objective_mlp(trial):
    tf.keras.backend.clear_session()
    params = {
        "n_layers": trial.suggest_int("n_layers", 1, 4),
        "activation": trial.suggest_categorical("activation", ["relu", "swish", "elu"]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.40),
        "batch_norm": trial.suggest_categorical("batch_norm", [True, False]),
        "l2": trial.suggest_float("l2", 1e-7, 1e-2, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024]),
    }
    for i in range(params["n_layers"]):
        params[f"units_{i}"] = trial.suggest_int(f"units_{i}", 32, 320, step=32)

    model = build_mlp(params, X_mlp_train.shape[1], SEED_BASE)
    early = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        min_delta=1e-5,
    )
    hist = model.fit(
        X_mlp_train,
        y_opt_train.to_numpy(),
        validation_data=(X_mlp_val, y_opt_val.to_numpy()),
        epochs=250,
        batch_size=params["batch_size"],
        verbose=0,
        callbacks=[early],
    )
    best_epoch = int(np.argmin(hist.history["val_loss"]) + 1)
    trial.set_user_attr("best_epoch", best_epoch)

    pred = model.predict(X_mlp_val, verbose=0).ravel()
    rmse = float(np.sqrt(mean_squared_error(y_opt_val, pred)))
    tf.keras.backend.clear_session()
    gc.collect()
    return rmse


study_mlp = optuna.create_study(
    study_name="mlp_pressao_poco",
    direction="minimize",
    sampler=TPESampler(seed=SEED_BASE),
    pruner=MedianPruner(n_startup_trials=5),
    storage=STORAGE,
    load_if_exists=True,
)
study_mlp = optimize_until_total(study_mlp, objective_mlp, N_TRIALS_MLP)

print("Melhor RMSE MLP:", study_mlp.best_value)
print("Melhores parâmetros MLP:")
print(study_mlp.best_params)
print("Melhor época:", study_mlp.best_trial.user_attrs.get("best_epoch"))

## 10. Resumo da otimização

In [ ]:
studies = {
    "Random Forest": study_rf,
    "XGBoost": study_xgb,
    "Rede Neural": study_mlp,
}

optimization_summary = pd.DataFrame([
    {
        "modelo": name,
        "melhor_RMSE_validacao": study.best_value,
        "melhor_trial": study.best_trial.number + 1,
        "melhores_parametros": json.dumps(study.best_params, ensure_ascii=False),
    }
    for name, study in studies.items()
]).sort_values("melhor_RMSE_validacao")

display(optimization_summary)
optimization_summary.to_csv(PROJECT_DIR / "resumo_otimizacao_5runs.csv", index=False)

best_params_export = {
    "Random Forest": study_rf.best_params,
    "XGBoost": {
        **study_xgb.best_params,
        "best_iteration": study_xgb.best_trial.user_attrs.get("best_iteration"),
    },
    "Rede Neural": {
        **study_mlp.best_params,
        "best_epoch": study_mlp.best_trial.user_attrs.get("best_epoch"),
    },
    "TabPFN-3": {
        "optuna": False,
        "observacao": "Resultados importados do experimento TabPFN-3 já executado, preservando a metodologia anterior.",
    },
}

with open(PROJECT_DIR / "melhores_hiperparametros_5runs.json", "w", encoding="utf-8") as f:
    json.dump(best_params_export, f, ensure_ascii=False, indent=2)

print("Arquivos de otimização salvos.")

## 11. Figura de convergência do Optuna

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
})

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True)
panel_labels = ["(a)", "(b)", "(c)"]

for ax, panel, (model_name, study) in zip(axes, panel_labels, studies.items()):
    completed_trials = [
        t for t in study.trials
        if t.state == TrialState.COMPLETE and t.value is not None
    ]
    completed_trials = completed_trials[:50]
    trial_numbers = np.array([i + 1 for i in range(len(completed_trials))])
    rmse_values = np.array([t.value for t in completed_trials])
    best_so_far = np.minimum.accumulate(rmse_values)

    ax.scatter(trial_numbers, rmse_values, s=16, alpha=0.55)
    ax.plot(trial_numbers, best_so_far, linewidth=1.8)

    best_idx = int(np.argmin(rmse_values))
    ax.scatter(trial_numbers[best_idx], rmse_values[best_idx], marker="*", s=75, zorder=5)
    ax.annotate(
        f"{rmse_values[best_idx]:.4f}",
        (trial_numbers[best_idx], rmse_values[best_idx]),
        xytext=(5, 8),
        textcoords="offset points",
        fontsize=8,
    )

    ax.set_title(f"{panel} {model_name}")
    ax.set_xlabel("Number of function evaluations")
    ax.set_ylabel("RMSE")
    ax.set_xlim(0, 51)
    ax.grid(alpha=0.18, linewidth=0.5)

fig.legend(
    ["Observed RMSE", "Best RMSE", "Best solution"],
    loc="upper center",
    bbox_to_anchor=(0.5, 1.07),
    ncol=3,
    frameon=False,
)
plt.tight_layout()
plt.savefig(PROJECT_DIR / "Fig_Optuna_Convergence_5runs.png", dpi=600, bbox_inches="tight")
plt.savefig(PROJECT_DIR / "Fig_Optuna_Convergence_5runs.pdf", bbox_inches="tight")
plt.show()

## 12. Treinamento final de RF, XGBoost e MLP — 5 repetições

In [ ]:
def fit_predict_rf(seed):
    params = dict(study_rf.best_params)
    params.update({"random_state": seed, "n_jobs": -1})
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", RandomForestRegressor(**params)),
    ])
    model.fit(X_all, y_all)
    return model.predict(X_test), model


def fit_predict_xgb(seed):
    params = dict(study_xgb.best_params)
    best_iteration = study_xgb.best_trial.user_attrs.get("best_iteration")
    if best_iteration is not None:
        params["n_estimators"] = int(best_iteration) + 1

    params.update({
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": XGB_DEVICE,
        "random_state": seed,
        "n_jobs": -1,
    })
    model = XGBRegressor(**params)
    model.fit(X_all, y_all, verbose=False)
    return model.predict(X_test), model


def fit_predict_mlp(seed):
    params = dict(study_mlp.best_params)
    best_epoch = int(study_mlp.best_trial.user_attrs.get("best_epoch", 100))

    preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])
    X_train_nn = preprocessor.fit_transform(X_all)
    X_test_nn = preprocessor.transform(X_test)

    tf.keras.backend.clear_session()
    model = build_mlp(params, X_train_nn.shape[1], seed)
    model.fit(
        X_train_nn,
        y_all.to_numpy(),
        epochs=best_epoch,
        batch_size=params["batch_size"],
        verbose=0,
    )
    pred = model.predict(X_test_nn, verbose=0).ravel()
    return pred, (preprocessor, model)


fitters = {
    "Random Forest": fit_predict_rf,
    "XGBoost": fit_predict_xgb,
    "Rede Neural": fit_predict_mlp,
}

results_ml = []
best_predictions = {}
best_models = {}

for model_name, fitter in fitters.items():
    print(f"\n===== {model_name} =====")
    best_rmse = np.inf

    for repetition, seed in enumerate(SEEDS_RUNS, start=1):
        print(f"{model_name} | repetição {repetition}/{N_REPETITIONS} | seed {seed}")
        set_all_seeds(seed)

        pred, fitted_model = fitter(seed)
        metrics = regression_metrics(y_test, pred, groups_test)
        row = {
            "modelo": model_name,
            "repeticao": repetition,
            "seed": seed,
            **metrics,
        }
        results_ml.append(row)
        print(row)

        if metrics["RMSE"] < best_rmse:
            best_rmse = metrics["RMSE"]
            best_predictions[model_name] = np.asarray(pred).copy()
            if model_name in best_models:
                del best_models[model_name]
            best_models[model_name] = fitted_model
        else:
            del fitted_model

        gc.collect()
        if model_name == "Rede Neural":
            try:
                tf.keras.backend.clear_session()
            except Exception:
                pass

        # checkpoint após cada repetição
        pd.DataFrame(results_ml).to_csv(
            PROJECT_DIR / "resultados_5_repeticoes_RF_XGB_MLP.csv",
            index=False,
        )

results_ml_df = (
    pd.DataFrame(results_ml)
    .sort_values(["modelo", "repeticao"])
    .reset_index(drop=True)
)
display(results_ml_df)
print("✅ RF, XGBoost e MLP concluídos nas 5 seeds.")

## 13. Importar os resultados reais do TabPFN-3

In [ ]:
# O notebook NÃO inventa nem completa resultados do TabPFN.
# Ele procura um CSV real já gerado no Colab/VS Code e usa somente as seeds 42–46.

tabpfn_candidate_dirs = [BASE_DIR, PROJECT_DIR, PROJECT_DIR.parent]
tabpfn_names = [
    "resultados_tabpfn_5_repeticoes.csv",
    "resultados_tabpfn_10_repeticoes.csv",
]

TABPFN_RESULTS_PATH = None
for folder in tabpfn_candidate_dirs:
    for name in tabpfn_names:
        p = folder / name
        if p.exists():
            TABPFN_RESULTS_PATH = p.resolve()
            break
    if TABPFN_RESULTS_PATH is not None:
        break

if TABPFN_RESULTS_PATH is None:
    raise FileNotFoundError(
        "Não encontrei o CSV do TabPFN. Coloque resultados_tabpfn_5_repeticoes.csv "
        "ou resultados_tabpfn_10_repeticoes.csv na pasta do notebook ou dos dados."
    )

print("CSV TabPFN encontrado:", TABPFN_RESULTS_PATH)
tabpfn_results = pd.read_csv(TABPFN_RESULTS_PATH)

# Normalização de nomes de coluna, se necessário.
if "R²" in tabpfn_results.columns and "R2" not in tabpfn_results.columns:
    tabpfn_results = tabpfn_results.rename(columns={"R²": "R2"})

# Mantém apenas as cinco seeds planejadas.
tabpfn_results = tabpfn_results[
    tabpfn_results["seed"].astype(int).isin(SEEDS_RUNS)
].copy()
tabpfn_results["seed"] = tabpfn_results["seed"].astype(int)
tabpfn_results = tabpfn_results.sort_values("seed").drop_duplicates("seed", keep="last")

print("Seeds reais encontradas no TabPFN:", tabpfn_results["seed"].tolist())
display(tabpfn_results)

expected = set(SEEDS_RUNS)
found = set(tabpfn_results["seed"].tolist())
missing = sorted(expected - found)
if missing:
    raise ValueError(
        f"O CSV do TabPFN ainda não possui as 5 execuções reais. Faltam as seeds: {missing}. "
        "O código não criará valores artificiais."
    )

required_cols = [
    "modelo", "repeticao", "seed", "RMSE", "MAE", "R2",
    "RMSE_medio_por_curva", "RMSE_mediano_por_curva",
]
missing_cols = [c for c in required_cols if c not in tabpfn_results.columns]
if missing_cols:
    raise ValueError(f"Colunas ausentes no CSV do TabPFN: {missing_cols}")

tabpfn_results = tabpfn_results[required_cols].copy()
print("✅ TabPFN: 5 resultados reais recuperados.")

## 14. Juntar os quatro modelos e calcular média ± desvio-padrão

In [ ]:
results_df = pd.concat(
    [results_ml_df, tabpfn_results],
    ignore_index=True,
    sort=False,
)
results_df = results_df.sort_values(["modelo", "repeticao"]).reset_index(drop=True)

print("Quantidade de execuções por modelo:")
print(results_df.groupby("modelo").size())

display(results_df)
results_df.to_csv(PROJECT_DIR / "resultados_5_repeticoes.csv", index=False)

summary_5 = (
    results_df.groupby("modelo")
    .agg(
        n=("RMSE", "size"),
        RMSE_medio=("RMSE", "mean"),
        RMSE_desvio=("RMSE", "std"),
        RMSE_melhor=("RMSE", "min"),
        MAE_medio=("MAE", "mean"),
        MAE_desvio=("MAE", "std"),
        R2_medio=("R2", "mean"),
        R2_desvio=("R2", "std"),
        RMSE_curva_medio=("RMSE_medio_por_curva", "mean"),
        RMSE_curva_desvio=("RMSE_medio_por_curva", "std"),
    )
    .sort_values(["RMSE_medio", "RMSE_curva_medio"])
)

display(summary_5)
summary_5.to_csv(PROJECT_DIR / "ranking_modelos_5_repeticoes.csv")

winner = summary_5.index[0]
print("🏆 Modelo vencedor pelo menor RMSE médio:", winner)

# Tabela pronta para o artigo.
rows = []
for model_name, group in results_df.groupby("modelo"):
    rows.append({
        "Model": model_name,
        "n": len(group),
        "RMSE": f"{group['RMSE'].mean():.4f} ± {group['RMSE'].std():.4f}",
        "MAE": f"{group['MAE'].mean():.4f} ± {group['MAE'].std():.4f}",
        "R²": f"{group['R2'].mean():.4f} ± {group['R2'].std():.4f}",
        "Mean RMSE per curve": (
            f"{group['RMSE_medio_por_curva'].mean():.4f} ± "
            f"{group['RMSE_medio_por_curva'].std():.4f}"
        ),
    })

table_article = pd.DataFrame(rows)
display(table_article)
table_article.to_excel(PROJECT_DIR / "Table_Model_Performance_5runs.xlsx", index=False)

## 15. Boxplot do RMSE — 5 repetições

In [ ]:
model_order = summary_5.index.tolist()
plot_data = [
    results_df.loc[results_df["modelo"] == model, "RMSE"].dropna().to_numpy()
    for model in model_order
]
labels = [f"{model}\n(n={len(values)})" for model, values in zip(model_order, plot_data)]

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.boxplot(plot_data, tick_labels=labels, showmeans=True)

rng = np.random.default_rng(42)
for i, values in enumerate(plot_data, start=1):
    jitter = rng.normal(i, 0.035, size=len(values))
    ax.scatter(jitter, values, s=22, alpha=0.7)

ax.set_ylabel("RMSE")
ax.set_xlabel("Machine learning model")
ax.grid(axis="y", alpha=0.20)
plt.tight_layout()
plt.savefig(PROJECT_DIR / "Fig_RMSE_5_repetitions.png", dpi=600, bbox_inches="tight")
plt.savefig(PROJECT_DIR / "Fig_RMSE_5_repetitions.pdf", bbox_inches="tight")
plt.show()

## 16. Salvar as melhores predições disponíveis e gráfico real × previsto

In [ ]:
# As predições individuais ficam disponíveis para RF/XGBoost/MLP porque foram executados neste notebook.
# Para o TabPFN, este notebook importou apenas as métricas. Se ele fosse o vencedor,
# seria necessário também importar o arquivo de predições da melhor execução do TabPFN.

predictions_out = test_raw.copy()
for model_name, pred in best_predictions.items():
    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("ã", "a")
        .replace("á", "a")
        .replace("ó", "o")
    )
    predictions_out[f"pred_{safe_name}"] = pred

predictions_out.to_excel(PROJECT_DIR / "predicoes_melhores_modelos_5runs.xlsx", index=False)

if winner in best_predictions:
    best_pred_winner = best_predictions[winner]

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.scatter(y_test, best_pred_winner, s=8, alpha=0.35)
    lo = min(float(y_test.min()), float(best_pred_winner.min()))
    hi = max(float(y_test.max()), float(best_pred_winner.max()))
    ax.plot([lo, hi], [lo, hi], linestyle="--")
    ax.set_xlabel("Observed pressure")
    ax.set_ylabel("Predicted pressure")
    ax.set_title(f"Observed × predicted — {winner}")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PROJECT_DIR / "Fig_Observed_vs_Predicted_Winner.png", dpi=600, bbox_inches="tight")
    plt.savefig(PROJECT_DIR / "Fig_Observed_vs_Predicted_Winner.pdf", bbox_inches="tight")
    plt.show()
else:
    print(
        f"O vencedor foi {winner}, mas suas predições individuais não foram carregadas neste notebook. "
        "Importe o arquivo de predições do TabPFN antes de gerar o gráfico real × previsto."
    )

## 17. Curvas representativas do melhor modelo

In [ ]:
if winner in best_predictions:
    pred_winner = best_predictions[winner]

    # Calcula o RMSE de cada curva para selecionar exemplos baixo/mediano/alto erro.
    temp_eval = pd.DataFrame({
        "curve_id": groups_test.to_numpy(),
        "sq_error": (y_test.to_numpy() - pred_winner) ** 2,
    })
    curve_rmse = np.sqrt(temp_eval.groupby("curve_id")["sq_error"].mean()).sort_values()

    selected_curves = [
        curve_rmse.index[0],
        curve_rmse.index[len(curve_rmse)//2],
        curve_rmse.index[-1],
    ]
    labels_case = ["Low error", "Median error", "High error"]

    for cid, case_name in zip(selected_curves, labels_case):
        mask = test_df[GROUP_COL].eq(cid)
        order = np.argsort(test_df.loc[mask, ORDER_COL].to_numpy())
        x = test_df.loc[mask, TIME_COL].to_numpy()[order]
        y_real = y_test.loc[mask].to_numpy()[order]
        y_pred = pred_winner[mask.to_numpy()][order]

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(x, y_real, marker="o", label="Observed")
        ax.plot(x, y_pred, marker="x", label="Predicted")
        ax.set_xlabel("Q_time")
        ax.set_ylabel("R_pressure")
        ax.set_title(f"{case_name} — curve {cid} — {winner}")
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        safe_case = case_name.lower().replace(" ", "_")
        plt.savefig(PROJECT_DIR / f"Fig_Curve_{safe_case}_{cid}.png", dpi=600, bbox_inches="tight")
        plt.show()
else:
    print("Curvas não geradas porque as predições do modelo vencedor não estão disponíveis neste notebook.")

In [ ]:
import optuna

DB_PATH = "optuna_pressao_poco.db"
STORAGE = f"sqlite:///{DB_PATH}"

estudos = optuna.study.get_all_study_summaries(storage=STORAGE)

print("ESTUDOS ENCONTRADOS:\n")

for estudo in estudos:
    print(
        f"Nome: {estudo.study_name} | "
        f"Nº de trials: {estudo.n_trials} | "
        f"Melhor valor: {estudo.best_trial.value:.6f}"
    )

In [ ]:
import optuna
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# CONFIGURAÇÕES
# =========================================================
DB_PATH = "optuna_pressao_poco.db"
STORAGE = f"sqlite:///{DB_PATH}"

# =========================================================
# LOCALIZAR ESTUDOS
# =========================================================
summaries = optuna.study.get_all_study_summaries(storage=STORAGE)

study_names = [s.study_name for s in summaries]

print("Estudos encontrados:")
for name in study_names:
    print("-", name)


def localizar_estudo(palavras):
    for nome in study_names:
        nome_lower = nome.lower()

        if any(p.lower() in nome_lower for p in palavras):
            return nome

    return None


rf_name = localizar_estudo([
    "random_forest",
    "randomforest",
    "random forest",
    "rf"
])

xgb_name = localizar_estudo([
    "xgboost",
    "xgb"
])

mlp_name = localizar_estudo([
    "mlp",
    "neural",
    "rede"
])


print("\nEstudos identificados:")
print("Random Forest:", rf_name)
print("XGBoost:", xgb_name)
print("MLP:", mlp_name)


if rf_name is None or xgb_name is None or mlp_name is None:
    raise ValueError(
        "Não consegui identificar automaticamente todos os estudos. "
        "Veja os nomes impressos acima e me envie."
    )


studies = {
    "Random Forest": optuna.load_study(
        study_name=rf_name,
        storage=STORAGE
    ),

    "XGBoost": optuna.load_study(
        study_name=xgb_name,
        storage=STORAGE
    ),

    "MLP": optuna.load_study(
        study_name=mlp_name,
        storage=STORAGE
    )
}

# =========================================================
# ESTILO
# =========================================================
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

fig, axes = plt.subplots(
    1,
    3,
    figsize=(13, 4.2)
)

letras = ["(a)", "(b)", "(c)"]

# =========================================================
# PLOT
# =========================================================
for ax, (modelo, study), letra in zip(
    axes,
    studies.items(),
    letras
):

    trials_validos = [
        t for t in study.trials
        if (
            t.state == optuna.trial.TrialState.COMPLETE
            and t.value is not None
            and np.isfinite(t.value)
        )
    ]

    numeros = np.array([
        t.number + 1
        for t in trials_validos
    ])

    valores = np.array([
        t.value
        for t in trials_validos
    ])

    # Melhor valor acumulado
    melhores = np.minimum.accumulate(valores)

    # Melhor trial
    idx_best = np.argmin(valores)

    best_trial = numeros[idx_best]
    best_value = valores[idx_best]

    # Todos os trials
    ax.scatter(
        numeros,
        valores,
        s=30,
        alpha=0.70,
        label="Validation RMSE"
    )

    # Melhor RMSE acumulado
    ax.plot(
        numeros,
        melhores,
        linewidth=2,
        label="Best RMSE"
    )

    # Destacar melhor trial
    ax.scatter(
        best_trial,
        best_value,
        s=75,
        marker="*",
        zorder=5
    )

    ax.annotate(
        f"Best = {best_value:.4f}\nTrial {best_trial}",
        xy=(best_trial, best_value),
        xytext=(5, 12),
        textcoords="offset points",
        fontsize=9
    )

    ax.set_title(f"{letra} {modelo}")

    ax.set_xlabel("Optuna trial")
    ax.set_ylabel("Validation RMSE")

    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.5,
        alpha=0.35
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        frameon=False,
        fontsize=8
    )

# =========================================================
# SALVAR
# =========================================================
fig.tight_layout()

fig.savefig(
    "Figura_1_Convergencia_Optuna.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    "Figura_1_Convergencia_Optuna.pdf",
    bbox_inches="tight"
)

plt.show()

print("\nArquivos gerados:")
print("Figura_1_Convergencia_Optuna.png")
print("Figura_1_Convergencia_Optuna.pdf")

In [ ]:
import optuna
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# CONFIGURAÇÕES
# =========================================================
DB_PATH = "optuna_pressao_poco.db"
STORAGE = f"sqlite:///{DB_PATH}"

# =========================================================
# LOCALIZAR ESTUDOS
# =========================================================
summaries = optuna.study.get_all_study_summaries(storage=STORAGE)

study_names = [s.study_name for s in summaries]

print("Estudos encontrados:")
for name in study_names:
    print("-", name)


def localizar_estudo(palavras):
    for nome in study_names:
        nome_lower = nome.lower()

        if any(p.lower() in nome_lower for p in palavras):
            return nome

    return None


rf_name = localizar_estudo([
    "random_forest",
    "randomforest",
    "random forest",
    "rf"
])

xgb_name = localizar_estudo([
    "xgboost",
    "xgb"
])

mlp_name = localizar_estudo([
    "mlp",
    "neural",
    "rede"
])


print("\nEstudos identificados:")
print("Random Forest:", rf_name)
print("XGBoost:", xgb_name)
print("MLP:", mlp_name)


if rf_name is None or xgb_name is None or mlp_name is None:
    raise ValueError(
        "Não consegui identificar automaticamente todos os estudos. "
        "Veja os nomes impressos acima."
    )


studies = {
    "Random Forest": optuna.load_study(
        study_name=rf_name,
        storage=STORAGE
    ),

    "XGBoost": optuna.load_study(
        study_name=xgb_name,
        storage=STORAGE
    ),

    "MLP": optuna.load_study(
        study_name=mlp_name,
        storage=STORAGE
    )
}

# =========================================================
# ESTILO PARA ARTIGO CIENTÍFICO
# =========================================================
plt.rcParams.update({
    "font.family": "Times New Roman",

    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,

    "xtick.labelsize": 10,
    "ytick.labelsize": 10,

    "legend.fontsize": 9,

    "axes.linewidth": 0.8,

    # Qualidade da visualização no notebook
    "figure.dpi": 150,

    # Qualidade padrão ao salvar
    "savefig.dpi": 600
})

# Figura propositalmente grande.
# Depois ela pode ser reduzida no artigo mantendo nitidez.
fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

letras = ["(a)", "(b)", "(c)"]

# =========================================================
# PLOT
# =========================================================
for ax, (modelo, study), letra in zip(
    axes,
    studies.items(),
    letras
):

    trials_validos = [
        t for t in study.trials
        if (
            t.state == optuna.trial.TrialState.COMPLETE
            and t.value is not None
            and np.isfinite(t.value)
        )
    ]

    numeros = np.array([
        t.number + 1
        for t in trials_validos
    ])

    valores = np.array([
        t.value
        for t in trials_validos
    ])

    # Melhor valor acumulado
    melhores = np.minimum.accumulate(valores)

    # Melhor trial
    idx_best = np.argmin(valores)

    best_trial = numeros[idx_best]
    best_value = valores[idx_best]

    # Todos os trials
    ax.scatter(
        numeros,
        valores,
        s=34,
        alpha=0.75,
        label="Validation RMSE",
        rasterized=False
    )

    # Melhor RMSE acumulado
    ax.plot(
        numeros,
        melhores,
        linewidth=2.0,
        label="Best RMSE"
    )

    # Melhor trial
    ax.scatter(
        best_trial,
        best_value,
        s=100,
        marker="*",
        zorder=5
    )

    ax.annotate(
        f"Best = {best_value:.4f}\nTrial {best_trial}",
        xy=(best_trial, best_value),
        xytext=(6, 12),
        textcoords="offset points",
        fontsize=9
    )

    # Identificação dos painéis
    ax.set_title(
        f"{letra} {modelo}",
        pad=8
    )

    ax.set_xlabel("Optuna trial")
    ax.set_ylabel("Validation RMSE")

    # Grade discreta
    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.5,
        alpha=0.30
    )

    ax.set_axisbelow(True)

    # Visual mais limpo
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        frameon=False,
        loc="best"
    )

# =========================================================
# AJUSTE FINAL
# =========================================================
fig.tight_layout(
    pad=1.5,
    w_pad=2.0
)

# =========================================================
# SALVAR — PNG 600 DPI
# =========================================================
fig.savefig(
    "Figura_1_Convergencia_Optuna_600dpi.png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.08,
    facecolor="white"
)

# =========================================================
# SALVAR — PDF VETORIAL
# =========================================================
fig.savefig(
    "Figura_1_Convergencia_Optuna_Vetorial.pdf",
    bbox_inches="tight",
    pad_inches=0.08,
    facecolor="white"
)

# =========================================================
# SALVAR — SVG VETORIAL
# =========================================================
fig.savefig(
    "Figura_1_Convergencia_Optuna_Vetorial.svg",
    bbox_inches="tight",
    pad_inches=0.08,
    facecolor="white"
)

# =========================================================
# SALVAR — TIFF 600 DPI
# =========================================================
fig.savefig(
    "Figura_1_Convergencia_Optuna_600dpi.tiff",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.08,
    facecolor="white"
)

plt.show()

print("\nArquivos gerados com qualidade de publicação:")
print("1. Figura_1_Convergencia_Optuna_600dpi.png")
print("2. Figura_1_Convergencia_Optuna_Vetorial.pdf")
print("3. Figura_1_Convergencia_Optuna_Vetorial.svg")
print("4. Figura_1_Convergencia_Optuna_600dpi.tiff")

In [ ]:
for modelo, study in studies.items():
    best = study.best_trial

    print("\n" + "=" * 60)
    print(modelo)
    print("=" * 60)

    print(f"Melhor trial: {best.number + 1}")
    print(f"Melhor RMSE de validação: {best.value:.6f}")

    print("\nHiperparâmetros:")
    for parametro, valor in best.params.items():
        print(f"{parametro}: {valor}")

    if best.user_attrs:
        print("\nInformações adicionais:")
        for chave, valor in best.user_attrs.items():
            print(f"{chave}: {valor}")

In [ ]:
# XGBOOST
study_xgb = studies["XGBoost"]
best_xgb = study_xgb.best_trial

print("XGBOOST")
print("Best trial:", best_xgb.number + 1)
print("Best RMSE:", best_xgb.value)

for k, v in best_xgb.params.items():
    print(f"{k} = {v}")